In [4]:
%cd practicas/

/workspace/practicas


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [5]:
from pyspark.sql import SparkSession

spark = ( SparkSession.builder
         .appName("pr503")
         .master("spark://spark-master:7077")
         .getOrCreate()
         )
 
sc = spark.sparkContext

In [6]:
from pyspark.sql.types import StructType, StructField, BooleanType, IntegerType, StringType, DoubleType, LongType, TimestampType
from pyspark.sql import functions as f
from pyspark.sql import Window

schema_world = StructType([
    StructField("row_id", IntegerType(), True),
    StructField("click_datetime", TimestampType(), True),
    StructField("time_to_next_click", DoubleType(), True),
    StructField("movie_title", StringType(), True),
    StructField("movie_genres", StringType(), True),
    StructField("release_date", StringType(), True),
    StructField("title_id", StringType(), True),
    StructField("user_id", StringType(), True),
    ])

df = (spark.read
             .format("csv")
             .schema(schema_world)
             .option("header", "True")
             .load("./data/vodclickstream_uk_movies_03.csv"))
df.show(5)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/workspace/data/vodclickstream_uk_movies_03.csv.

26/03/18 11:23:08 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


# 1. Auditoría de telemetría Web (validación de datos)

In [ ]:
ventana = ( Window.partitionBy("user_id").orderBy("click_datetime") )

df_resultado = df.select("user_id", "click_datetime").withColumn(
    "click_anterior", f.lead("click_datetime", -1).over(ventana)
).withColumn("calculated_time_to_next", (f.col("click_anterior").cast("long") - f.col("click_datetime").cast("long"))*(-1))

df_resultado.show(5)

+----------+-------------------+-------------------+-----------------------+
|   user_id|     click_datetime|     click_anterior|calculated_time_to_next|
+----------+-------------------+-------------------+-----------------------+
|0006ea6b5c|2017-05-19 20:21:43|               NULL|                   NULL|
|0006ea6b5c|2017-05-20 21:54:34|2017-05-19 20:21:43|                  91971|
|0006ea6b5c|2017-05-26 18:38:01|2017-05-20 21:54:34|                 506607|
|0006ea6b5c|2017-05-26 23:31:46|2017-05-26 18:38:01|                  17625|
|0006ea6b5c|2017-05-27 22:45:41|2017-05-26 23:31:46|                  83635|
+----------+-------------------+-------------------+-----------------------+
only showing top 5 rows



# 2. Deteccion de "zapping"

In [ ]:
df_resultado = df_resultado.withColumn("es_zaping", f.when((f.col("click_anterior").cast("long") - f.col("click_datetime").cast("long"))*(-1) > 300 , "NO").otherwise("SI"))
df_resultado.show(5)

+----------+-------------------+-------------------+-----------------------+---------+
|   user_id|     click_datetime|     click_anterior|calculated_time_to_next|es_zaping|
+----------+-------------------+-------------------+-----------------------+---------+
|0006ea6b5c|2017-05-19 20:21:43|               NULL|                   NULL|       SI|
|0006ea6b5c|2017-05-20 21:54:34|2017-05-19 20:21:43|                  91971|       NO|
|0006ea6b5c|2017-05-26 18:38:01|2017-05-20 21:54:34|                 506607|       NO|
|0006ea6b5c|2017-05-26 23:31:46|2017-05-26 18:38:01|                  17625|       NO|
|0006ea6b5c|2017-05-27 22:45:41|2017-05-26 23:31:46|                  83635|       NO|
+----------+-------------------+-------------------+-----------------------+---------+
only showing top 5 rows



# 3. El rankinf de "maratones"

In [ ]:
df_resultado = df_resultado.withColumn("click_date", f.split(f.col("click_datetime"), " ")[0])



ventana = (Window.partitionBy("user_Id", "click_date").orderBy("click_date"))

df_resultado = df_resultado.withColumn("row_number", f.row_number().over(ventana))

df_resultado.filter(f.col("row_number") >= 5).sort("row_number", ascending=True).show(5)

+----------+-------------------+-------------------+-----------------------+---------+----------+----------+
|   user_id|     click_datetime|     click_anterior|calculated_time_to_next|es_zaping|click_date|row_number|
+----------+-------------------+-------------------+-----------------------+---------+----------+----------+
|0060324ef4|2017-09-29 22:07:05|2017-09-29 21:57:08|                    597|       NO|2017-09-29|         5|
|00c10cd87c|2017-10-24 15:10:13|2017-10-24 15:03:39|                    394|       NO|2017-10-24|         5|
|010ce0857b|2019-05-01 18:33:30|2019-05-01 18:33:30|                      0|       SI|2019-05-01|         5|
|00f86ba072|2019-06-28 20:32:50|2019-06-28 19:02:08|                   5442|       NO|2019-06-28|         5|
|0029f6bb1e|2017-01-21 17:15:00|2017-01-21 15:14:38|                   7222|       NO|2017-01-21|         5|
+----------+-------------------+-------------------+-----------------------+---------+----------+----------+
only showing top 5 

# 4. Análisis de re-visualización

In [ ]:
ventana = Window.partitionBy("user_id", "title_id")

df = df.withColumn("veces_vista_por_usuario", f.count("user_id").over(ventana))

df.filter(f.col("veces_vista_por_usuario") >= 3).show(10
)

+------+-------------------+------------------+--------------------+--------------------+-------------+----------+----------+-----------------------+
|row_id|     click_datetime|time_to_next_click|         movie_title|        movie_genres| release_date|  title_id|   user_id|veces_vista_por_usuario|
+------+-------------------+------------------+--------------------+--------------------+-------------+----------+----------+-----------------------+
|438764|2018-06-15 02:51:15|               0.0|From Dusk till Da...|       NOT AVAILABLE|NOT AVAILABLE|4c3d7b724e|000118a755|                      3|
|438844|2018-06-15 03:01:15|               0.0|From Dusk till Da...|       NOT AVAILABLE|NOT AVAILABLE|4c3d7b724e|000118a755|                      3|
|438863|2018-06-15 03:01:15|              -1.0|From Dusk till Da...|       NOT AVAILABLE|NOT AVAILABLE|4c3d7b724e|000118a755|                      3|
|578906|2018-12-30 22:05:13|               0.0|Black Mirror: Ban...|Drama, Mystery, S...|   2018-12-

26/03/05 10:03:21 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
